In [2]:
import os

os.listdir("/home/jovyan/drivers")

['postgresql.jar']

In [2]:
import os
import logging
import psycopg2

from pyspark.sql import SparkSession

# =========================================================
# Environment Variables
# =========================================================

POSTGRES_HOST = os.getenv("POSTGRES_OLIST_HOST")
POSTGRES_PORT = os.getenv("POSTGRES_OLIST_PORT", "5432")
POSTGRES_DB = os.getenv("POSTGRES_OLIST_DB")

POSTGRES_USER = os.getenv("POSTGRES_OLIST_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_OLIST_PASSWORD")

DATA_PATH = "/home/jovyan/data/olist"

# =========================================================
# Logging
# =========================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)

# =========================================================
# Validation
# =========================================================

required_vars = {
    "POSTGRES_OLIST_HOST": POSTGRES_HOST,
    "POSTGRES_OLIST_DB": POSTGRES_DB,
    "POSTGRES_OLIST_USER": POSTGRES_USER,
    "POSTGRES_OLIST_PASSWORD": POSTGRES_PASSWORD
}

missing = [k for k, v in required_vars.items() if not v]

if missing:
    raise ValueError(
        f"Variáveis obrigatórias não encontradas: {', '.join(missing)}"
    )

# =========================================================
# Spark Session
# =========================================================

spark = (
    SparkSession.builder
    .appName("olist_csv_to_postgres")
    .config(
        "spark.jars",
        "/home/jovyan/drivers/postgresql.jar"
    )
    .getOrCreate()
)

# =========================================================
# JDBC
# =========================================================

JDBC_URL = (
    f"jdbc:postgresql://{POSTGRES_HOST}:5432/{POSTGRES_DB}"
)

# =========================================================
# Utils
# =========================================================

def get_table_name(filename: str) -> str:
    """
    Exemplo:

    olist_orders_dataset.csv
    -> orders

    olist_order_items_dataset.csv
    -> order_items
    """

    name = filename.lower()

    name = name.replace(".csv", "")

    if name.startswith("olist_"):
        name = name[6:]

    name = name.replace("_dataset", "")

    return name


# =========================================================
# PostgreSQL
# =========================================================

def create_schema_if_not_exists():

    logger.info("Validando schema raw")

    conn = psycopg2.connect(
        host=POSTGRES_HOST,
        port=5432,
        database=POSTGRES_DB,
        user=POSTGRES_USER,
        password=POSTGRES_PASSWORD
    )

    conn.autocommit = True

    cursor = conn.cursor()

    cursor.execute(
        """
        CREATE SCHEMA IF NOT EXISTS raw;
        """
    )

    cursor.close()
    conn.close()

    logger.info("Schema raw pronto")


# =========================================================
# CSV -> PostgreSQL
# =========================================================

def load_csv_to_postgres(csv_path: str):

    file_name = os.path.basename(csv_path)

    table_name = get_table_name(file_name)

    logger.info(f"Lendo {file_name}")

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(csv_path)
    )

    record_count = df.count()

    logger.info(
        f"Gravando raw.{table_name} ({record_count:,} registros)"
    )

    (
        df.write
        .format("jdbc")
        .option("url", JDBC_URL)
        .option("dbtable", f"raw.{table_name}")
        .option("user", POSTGRES_USER)
        .option("password", POSTGRES_PASSWORD)
        .option("driver", "org.postgresql.Driver")
        .mode("overwrite")
        .save()
    )

    logger.info(
        f"Tabela raw.{table_name} carregada com sucesso"
    )


# =========================================================
# Main
# =========================================================

def main():

    logger.info("Iniciando carga dos CSVs da Olist")

    create_schema_if_not_exists()

    if not os.path.exists(DATA_PATH):

        logger.error(
            f"Pasta não encontrada: {DATA_PATH}"
        )

        return

    csv_files = sorted(
        [
            os.path.join(DATA_PATH, file)
            for file in os.listdir(DATA_PATH)
            if file.endswith(".csv")
        ]
    )

    if not csv_files:

        logger.warning(
            f"Nenhum CSV encontrado em {DATA_PATH}"
        )

        return

    logger.info(
        f"{len(csv_files)} arquivos encontrados"
    )

    for csv_file in csv_files:

        try:

            load_csv_to_postgres(csv_file)

        except Exception as e:

            logger.exception(
                f"Erro ao processar {csv_file}: {str(e)}"
            )

    logger.info("Carga concluída com sucesso")


# =========================================================
# Entry Point
# =========================================================

if __name__ == "__main__":
    main()

2026-06-05 17:37:36,221 - INFO - Iniciando carga dos CSVs da Olist
2026-06-05 17:37:36,221 - INFO - Validando schema raw
2026-06-05 17:37:36,227 - INFO - Schema raw pronto
2026-06-05 17:37:36,228 - INFO - 9 arquivos encontrados
2026-06-05 17:37:36,228 - INFO - Lendo olist_customers_dataset.csv
2026-06-05 17:37:36,422 - INFO - Gravando raw.customers (99,441 registros)
2026-06-05 17:37:37,160 - INFO - Tabela raw.customers carregada com sucesso
2026-06-05 17:37:37,160 - INFO - Lendo olist_geolocation_dataset.csv
2026-06-05 17:37:37,709 - INFO - Gravando raw.geolocation (1,000,163 registros)
2026-06-05 17:37:39,417 - INFO - Tabela raw.geolocation carregada com sucesso
2026-06-05 17:37:39,417 - INFO - Lendo olist_order_items_dataset.csv
2026-06-05 17:37:39,607 - INFO - Gravando raw.order_items (112,650 registros)
2026-06-05 17:37:40,126 - INFO - Tabela raw.order_items carregada com sucesso
2026-06-05 17:37:40,126 - INFO - Lendo olist_order_payments_dataset.csv
2026-06-05 17:37:40,318 - INFO

In [1]:
spark.stop()

NameError: name 'spark' is not defined